## Tahap 2 lanjutan — Finding 09 (sweep per-layer) + Finding 10 (probe readout)

Lanjutan finding 08 (patching v2): AGE (peta setia) → jembatan tersambung
di layer 11. RACE (peta rapuh) → TIDAK tersambung di layer 11, malah
1-head-bintang mendorong salah arah. Dua pertanyaan tersisa:

- **Finding 09** — kalau bukan L11, RACE menjawab dari mana? Sweep semua
  32 layer (semua 32 head tiap layer, satu layer per kondisi) → peta
  pengaruh-ke-output, disandingkan nanti (lokal) dengan peta kesetiaan
  (finding 06).
- **Finding 10** — berapa banyak informasi yang "hilang di koridor"?
  Latih probe kecil baca L11 (semua head + head bintang saja) di posisi
  token JAWABAN OPINI (bukan token identitas) → bandingkan akurasi probe
  vs akurasi mulut, dua-duanya lawan `group_real_dist`. Fitting probe
  dikerjakan LOKAL (CPU) dari fitur yang diekstrak di sini -- pola yang
  sama dengan notebook 10.

**Trik efisiensi (pelajaran dari notebook 12):** sweep 32 layer + 1
kontrol acak = 33 kondisi per (pasangan, soal). Alih-alih 33 forward pass
satu-satu, semuanya digabung jadi **1 batch** -- tiap item di batch itu
dipatch di 1 layer yang BEDA-BEDA sekaligus (diverifikasi offline lewat
dry-run sebelum notebook ini ditulis: patch per-item di 1 batch itu
saling terisolasi, tidak bocor ke item lain).


## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. Sesi bekas crash -> RESTART SESSION.
3. **Download setelah selesai** dari `/kaggle/working/tahap2_sweep_probe/`:
   `sweep_rows.csv`, `sweep_summary.csv`, `probe_features_L11.csv`,
   `probe_features_L11.npz` -> taruh di
   `notebooks/output/13_tahap2_sweep_dan_probe_kaggle/`.

Estimasi: load model ~10 menit; baseline batched ~10-15 menit; sweep
(720 forward batched, batch=33) ~15-20 menit. Total ~45 menit - 1 jam.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv tidak ketemu.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]  # sama dgn notebook 12
N_PAIRS = 12
N_QUESTIONS = 20      # per pasangan, top berdasarkan WD asli terbesar
MAX_OPTIONS = 6
MIN_SHARED_Q = 20

STAR_LAYER, STAR_HEAD = 11, 16   # buat referensi finding 10 (probe head-tunggal)
_forbidden_ctrl = {(STAR_LAYER, STAR_HEAD)}

OUT_DIR = "/kaggle/working/tahap2_sweep_probe"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pasangan + soal ber-perbedaan-asli terbesar

Persis logika notebook 12 (soal dipilih PER PASANGAN, top-N berdasar WD
distribusi ASLI -- coverage antar sel sparse, RACExRELIG nggak punya
pertanyaan yang dijawab semua sel).


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=sorted({c for p in pairs for c in p}), pairs=pairs,
                    pair_questions=pair_questions, needed=needed,
                    v1_opts=v1_opts, v2_opts=v2_opts)
    print(f"[{ty}] {len(plan[ty]['cells'])} sel, {len(pairs)} pasangan, baseline unik {len(needed)}")


## 2. Prompt QA-demografis + posisi token identitas & posisi jawaban-opini

Sama dengan notebook 12 (identitas = 1 token jawaban di blok QA demografis),
PLUS posisi baru: **token terakhir prompt** (persis posisi logits yang
sudah dipakai buat baca jawaban opini) -- ini yang dipakai finding 10
sebagai "state model tepat sebelum menjawab", buat dilatih probe-nya.


In [ ]:
ATTR_QA = {
    "RACExRELIG":       ("What is this survey respondent's race?",
                         "What is this survey respondent's religion?"),
    "RELIGxPOLPARTY":   ("What is this survey respondent's religion?",
                         "What is this survey respondent's political party affiliation?"),
    "AGExPOLPARTY":     ("What is this survey respondent's age group?",
                         "What is this survey respondent's political party affiliation?"),
}
DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    p = plan[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, p["v1_opts"], v1), "", demo_block(q2, p["v2_opts"], v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def full_positions(prompt):
    """Posisi 2 token identitas + posisi token TERAKHIR (jawaban opini)."""
    positions = []
    search_from = 0
    for _ in range(2):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))  # +1 BOS
        positions.append(pos)
        search_from = idx + 1
    last_pos = 1 + len(tokenizer.encode(prompt, add_special_tokens=False)) - 1
    positions.append(last_pos)
    return positions  # [id_pos1, id_pos2, last_pos]

# validasi cepat: A vs B beda cuma di 2 token identitas, posisi (termasuk last_pos) sama
ty0 = TYPES_RUN[0]
(A0, B0) = plan[ty0]["pairs"][0]
qk0 = plan[ty0]["pair_questions"][(A0, B0)][0]
pA, pB = build_prompt(ty0, A0, qk0), build_prompt(ty0, B0, qk0)
tA = tokenizer(pA, return_tensors="pt")["input_ids"][0]
tB = tokenizer(pB, return_tensors="pt")["input_ids"][0]
posA, posB = full_positions(pA), full_positions(pB)
assert len(tA) == len(tB), (len(tA), len(tB))
assert posA == posB, (posA, posB)
assert posA[-1] == len(tA) - 1, "last_pos harus sama dgn indeks token terakhir"
diff = (tA != tB).nonzero().flatten().tolist()
assert set(diff).issubset(set(posA[:2])), "token beda bukan di posisi identitas!"
print("Posisi tervalidasi:", posA, "(2 identitas + 1 posisi jawaban-opini)")


## 3. Load model + mesin hook batch-index-aware

Beda dari notebook 12: patch sekarang **per-item-di-batch** (tiap item
boleh dipatch di layer berbeda), bukan cuma per-layer global. Sudah
diverifikasi lewat dry-run offline sebelum notebook ini ditulis.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS
ALL_HEADS = list(range(NUM_HEADS))

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, NUM_LAYERS)), int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden_ctrl:
        break
print("Head kontrol acak:", RAND_HEAD)

CAPTURE_LAYERS = list(range(NUM_LAYERS))  # semua layer -- dibutuhkan donor sweep finding 09

_donor_capture = {}   # layer -> {pos: tensor[B, hidden]} (fp16, hemat RAM)
_capture_positions = []
_active_patch = {}    # layer -> list of (batch_idx, pos, heads, alpha, donor_vec4096)

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().half().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (b, pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                if len(heads) == NUM_HEADS:
                    # optimasi: "semua head" = seluruh vektor -> 1 operasi,
                    # bukan loop 32x (ini yang bikin sweep lambat: ribuan
                    # kernel kecil terpisah, padahal matematis sama saja)
                    x[b, pos, :] = x[b, pos, :] + alpha * (d - x[b, pos, :])
                else:
                    for h in heads:
                        s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                        x[b, pos, s] = x[b, pos, s] + alpha * (d[s] - x[b, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print(f"Hook terpasang di {len(CAPTURE_LAYERS)} layer.")

@torch.no_grad()
def forward_batch(prompts, n_opt, capture_positions=None, patch_spec=None):
    """Satu forward utk banyak prompt (identik panjangnya). Bisa capture DAN/ATAU patch."""
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    _active_patch.clear()
    sel = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=1).cpu().numpy()


## 4. Pass 1 — baseline (batched per soal) + donor semua layer + fitur probe

Batching persis pola notebook 12 (kelompokkan per `qk`, gabung semua `gk`
jadi 1 batch -- prompt sama panjang, cuma beda 1-2 token huruf identitas).
Donor SEMUA 32 layer disimpan di memori (dipakai sweep di bawah, TIDAK
disimpan ke disk -- terlalu besar). Fitur probe (vektor L11 di posisi
jawaban-opini) diambil sekarang, disimpan terpisah (kecil, buat finding 10).


In [ ]:
BASELINE_BATCH_SIZE = 16

baseline_pred = {}
donors = {}      # (ty, gk, qk) -> {layer: {pos: vec}}  (di memori, utk sweep)
id_pos = {}      # (ty, qk) -> [id_pos1, id_pos2, last_pos]
probe_rows = []  # (ty, gk, qk, n_opt, mouth_pred) -- metadata utk finding 10
probe_vecs = []  # vektor L11 4096-dim di posisi last_pos, sejajar dgn probe_rows

for ty in TYPES_RUN:
    p = plan[ty]
    by_qk = {}
    for (gk, qk) in p["needed"]:
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baseline {ty}"):
        prompts = [build_prompt(ty, gk, qk) for gk in gks]
        if (ty, qk) not in id_pos:
            id_pos[(ty, qk)] = full_positions(prompts[0])
        positions = id_pos[(ty, qk)]
        n_opt = len(qmeta[qk][2])
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]

        if len(set(lens)) != 1:
            print(f"  [{ty}/{qk}] panjang token tidak seragam {set(lens)} -> skip (jarang terjadi)")
            continue

        for start in range(0, len(gks), BASELINE_BATCH_SIZE):
            batch_gks = gks[start:start + BASELINE_BATCH_SIZE]
            batch_prompts = prompts[start:start + BASELINE_BATCH_SIZE]
            preds = forward_batch(batch_prompts, n_opt, capture_positions=positions)
            for b, gk in enumerate(batch_gks):
                baseline_pred[(gk, qk)] = preds[b]
                donors[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                    for L in CAPTURE_LAYERS
                }
                last_pos = positions[-1]
                vec_l11 = donors[(ty, gk, qk)][STAR_LAYER][last_pos].float().numpy()
                probe_rows.append(dict(ty=ty, gk=gk, qk=qk, n_opt=n_opt,
                                       mouth_pred=",".join(f"{x:.6f}" for x in preds[b])))
                probe_vecs.append(vec_l11)

print(f"{len(baseline_pred)} baseline selesai. {len(probe_rows)} fitur probe terkumpul.")


## 5. Finding 09 — sweep 32 layer + 1 kontrol acak, DIGABUNG jadi 1 batch

Per (pasangan, soal): batch berisi 33 salinan prompt A, tiap item dipatch
di **1 layer berbeda** (semua 32 head di layer itu) pakai donor B, item
ke-33 dipatch di 1 head acak (kontrol). Satu forward pass -> 33 kondisi
sekaligus.


In [ ]:
def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

sweep_rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for (A, B) in tqdm(p["pairs"], desc=f"sweep {ty}"):
        for qk in p["pair_questions"][(A, B)]:
            if (ty, qk) not in id_pos or (ty, A, qk) not in donors or (ty, B, qk) not in donors:
                continue
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            positions = id_pos[(ty, qk)][:2]  # 2 posisi identitas (bukan last_pos)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(ty, A, qk)

            n_cond = NUM_LAYERS + 1  # 32 layer + 1 kontrol acak
            batch_prompts = [prompt_A] * n_cond
            spec = {}
            for L in range(NUM_LAYERS):
                spec.setdefault(L, [])
                for pos in positions:
                    spec[L].append((L, pos, ALL_HEADS, 1.0, donors[(ty, B, qk)][L][pos]))
            ctrl_idx = NUM_LAYERS
            rl, rh = RAND_HEAD
            spec.setdefault(rl, [])
            for pos in positions:
                spec[rl].append((ctrl_idx, pos, [rh], 1.0, donors[(ty, B, qk)][rl][pos]))

            preds = forward_batch(batch_prompts, n_opt, patch_spec=spec)

            # dihitung SEKALI per (pasangan, soal) -- tidak tergantung layer,
            # jadi jangan diulang 32x di dalam loop L di bawah (percuma)
            wd_ctrl_realB = wd(preds[ctrl_idx], realB, ordinal)
            wd_A_realA = wd(predA, realA, ordinal)
            wd_A_realB = wd(predA, realB, ordinal)
            wd_B_realB = wd(predB, realB, ordinal)
            for L in range(NUM_LAYERS):
                sweep_rows.append(dict(
                    attr_type=ty, pair=f"{A} -> {B}", qkey=qk, layer=L,
                    wd_A_to_realA=wd_A_realA,
                    wd_A_to_realB=wd_A_realB,
                    wd_B_to_realB=wd_B_realB,
                    wd_patch_to_realB=wd(preds[L], realB, ordinal),
                    wd_ctrl_to_realB=wd_ctrl_realB,
                ))

sweep = pd.DataFrame(sweep_rows)
sweep["shift_ke_realB"] = sweep["wd_A_to_realB"] - sweep["wd_patch_to_realB"]
sweep["shift_ctrl_ke_realB"] = sweep["wd_A_to_realB"] - sweep["wd_ctrl_to_realB"]
sweep.to_csv(os.path.join(OUT_DIR, "sweep_rows.csv"), index=False)
print(sweep.shape, "-> sweep_rows.csv")


## 6. Ringkasan sweep: layer mana yang paling berpengaruh, per tipe

Bandingkan tiap layer lawan kontrol acak (dalam pasangan-soal yang SAMA,
jadi Wilcoxon berpasangan) -- fokus utama: **RACExRELIG, layer mana yang
"mengambil alih" peran L11?**


In [ ]:
summary_rows = []
print("=" * 100)
for ty in TYPES_RUN:
    s_ty = sweep[sweep["attr_type"] == ty]
    print(f"\n{ty} -- top 5 layer (mean shift_ke_realB):")
    per_layer = s_ty.groupby("layer").agg(
        mean_shift=("shift_ke_realB", "mean"),
        pct_pos=("shift_ke_realB", lambda x: (x > 0).mean()),
        n=("shift_ke_realB", "size"),
    ).reset_index()
    for L in per_layer["layer"]:
        row = s_ty[s_ty["layer"] == L]
        ctrl = row["shift_ctrl_ke_realB"]
        if len(row) > 10 and not np.allclose(row["shift_ke_realB"], ctrl):
            p = float(wilcoxon(row["shift_ke_realB"], ctrl).pvalue)
        else:
            p = float("nan")
        summary_rows.append(dict(attr_type=ty, layer=int(L),
                                 mean_shift=float(row["shift_ke_realB"].mean()),
                                 pct_positive=float((row["shift_ke_realB"] > 0).mean()),
                                 p_vs_random_ctrl=p))
    top5 = per_layer.sort_values("mean_shift", ascending=False).head(5)
    print(top5.to_string(index=False))

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "sweep_summary.csv"), index=False)
print("\n-> sweep_summary.csv (semua layer x tipe, dgn p vs kontrol acak)")


## 7. Simpan fitur probe (finding 10) -- vektor L11 di posisi jawaban-opini

Disimpan ringkas: metadata + prediksi mulut ke CSV, vektor 4096-dim ke
npz terpisah (biar CSV tetap kecil). Fitting probe dikerjakan LOKAL.


In [ ]:
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(os.path.join(OUT_DIR, "probe_features_L11.csv"), index=False)
np.savez_compressed(os.path.join(OUT_DIR, "probe_features_L11.npz"),
                    vecs=np.stack(probe_vecs).astype(np.float32))
print(probe_df.shape, "-> probe_features_L11.csv + probe_features_L11.npz")
print(f"HEAD_DIM={HEAD_DIM}, STAR_HEAD={STAR_HEAD} -> slice kolom "
      f"[{STAR_HEAD*HEAD_DIM}:{(STAR_HEAD+1)*HEAD_DIM}] dari vecs utk probe 1-head.")


## Cara baca hasil & checklist

**Finding 09 (sweep):**
- Cari layer dengan `mean_shift` tertinggi + `p_vs_random_ctrl` < 0.05 per
  tipe. Untuk AGE, harapannya L11 (atau dekat situ) tetap salah satu yang
  terbaik (validasi silang dgn finding 08). Untuk **RACE**, cari layer
  LAIN yang menang -- itu jawaban "model menjawab dari mana kalau bukan
  L11".
- Kalau TIDAK ADA layer yang signifikan buat RACE -- itu juga temuan:
  bukan salah lokasi, tapi identitas ras memang tidak dipakai kausal di
  attention manapun (mungkin lewat MLP, atau memang tidak dipakai sama
  sekali).

**Finding 10 (probe, dikerjakan lokal setelah download):**
- Latih ridge regression per soal: `vecs` (atau slice head-16 saja) ->
  distribusi ASLI, cross-validation leave-one-group-out (n kecil per
  soal). Skor WD probe vs `mouth_pred` (sudah ada di CSV), agregat per
  tipe. Bandingkan juga probe-32-head vs probe-1-head (starhead).

**Download (WAJIB):** `sweep_rows.csv`, `sweep_summary.csv`,
`probe_features_L11.csv`, `probe_features_L11.npz` ->
`notebooks/output/13_tahap2_sweep_dan_probe_kaggle/`.
